# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source

The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL and load the metadata
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

dataset = mlc.Dataset(croissant_url)

# Print the dataset metadata
md = dataset.metadata
print('Dataset Name:', md.name)
print('Description:', md.description)
print('Version:', getattr(md, 'version', None))
print('Published:', getattr(md, 'datePublished', None))
print('Identifier:', getattr(md, 'identifier', None))

# Print the dataset license and citation
print('License:', getattr(md, 'license', None))
print('Cite As:', getattr(md, 'citeAs', None))


## 2. Data Overview

Review available record sets, fields, and their IDs.

All entities are referenced by their `@id` fields. The following code prints available record sets, their `@id`s, and contained fields.

In [ ]:
# List all record sets, their IDs, and their fields
print('Available Record Sets:')
record_sets = list(dataset.record_sets)
for rset in record_sets:
    print(f"  Record set name: {rset.name}")
    print(f"    @id: {rset['@id']}")
    print(f"    Fields and columns:")
    for field in rset.fields:
        print(f"      Field: {field.name} (@id: {field['@id']}) Column(s): {[col['@id'] for col in field.columns]}")
    print("")
if not record_sets:
    print('No record sets found. Your dataset may not list explicit record sets, or record sets are defined on files directly.')

# Alternative: If no record sets, list files and their columns
if not record_sets:
    print('Checking for table-like files in dataset.distribution...')
    for dist in getattr(dataset.metadata, 'distribution', []):
        if hasattr(dist, 'columns'):
            print(f"  Table file name: {getattr(dist, 'name', None)} | @id: {getattr(dist, '@id', None)}")
            for col in dist.columns:
                print(f"    Column: {col.name} (@id: {col['@id']})")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis.

We use record set and field `@id`s from the overview section. 

If your dataset only contains one main record set, select its `@id`. Replace `<record_set_id>` with the correct `@id` value below.

In [ ]:
# If no explicit record sets, check dataset.distribution for tabular file objects
if not record_sets:
    # Get the main tabular file from distribution
    main_file = None
    for dist in getattr(dataset.metadata, 'distribution', []):
        if hasattr(dist, 'columns'):
            main_file = dist
            break
    if main_file:
        main_id = getattr(main_file, '@id', None)
        print(f"Selected table file: {main_id}")
    else:
        raise Exception('No tabular file object found in metadata.distribution.')
    # Load records using the file's @id
    records = list(dataset.records(record_set=main_id))
    df = pd.DataFrame(records)
    print(f"Columns in main table ({main_id}):\n", df.columns.tolist())
    df.head()
else:
    # For datasets with explicit record sets
    dataframes = {}
    record_set_ids = [rset['@id'] for rset in record_sets]
    for rs_id in record_set_ids:
        records = list(dataset.records(record_set=rs_id))
        dataframes[rs_id] = pd.DataFrame(records)
    print(f"Columns in first record set ({record_set_ids[0]}):\n", dataframes[record_set_ids[0]].columns.tolist())
    dataframes[record_set_ids[0]].head()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates removing outliers, transforming data distributions, and grouping data by key attributes using the field `@id`s.

In [ ]:
# Let's work with the main DataFrame (either df, if no explicit record sets, or the first in dataframes)
import numpy as np

if 'df' in locals():
    active_df = df
    print(f'Analyzing main tabular file.')
else:
    main_rs = list(dataframes.keys())[0]
    active_df = dataframes[main_rs]
    print(f'Analyzing record set with @id: {main_rs}')

# Show numeric columns by inferring types
numeric_cols = active_df.select_dtypes(include=[np.number]).columns.tolist()
if not numeric_cols:
    # Try to convert obvious float columns
    for col in active_df.columns:
        try:
            active_df[col] = pd.to_numeric(active_df[col])
        except Exception:
            pass
    numeric_cols = active_df.select_dtypes(include=[np.number]).columns.tolist()

print('Numeric columns:', numeric_cols)

# Select a demo numeric field for filtering, normalization
if numeric_cols:
    numeric_field = numeric_cols[0]  # Use the first numeric field
    threshold = active_df[numeric_field].mean() if active_df[numeric_field].notnull().any() else 0
    # Filtering rows where the selected field exceeds its mean value
    filtered_df = active_df[active_df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Attempt to group by another field (pick first non-numeric string field)
    group_fields = [col for col in active_df.columns if active_df[col].dtype == object and col != numeric_field]
    if group_fields:
        group_field = group_fields[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (mean of {numeric_field}):")
        print(grouped_df.head())
else:
    print('No numeric field found for filtering and EDA.')

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.

Below, we plot the distribution of the selected numeric field and, if a grouping field is available, visualize its aggregate means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

if numeric_cols:
    # Histogram of the numeric field
    plt.figure(figsize=(7, 4))
    sns.histplot(active_df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()
    
    # If we made a grouped_df, show barplot
    if 'grouped_df' in locals():
        plt.figure(figsize=(10, 5))
        sns.barplot(x=group_field, y=numeric_field, data=grouped_df)
        plt.title(f'Mean {numeric_field} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(f'Mean {numeric_field}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No numeric field found for visualization.')

## 6. Conclusion

In this notebook, we explored the Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells dataset using the `mlcroissant` library. We loaded the dataset's metadata and tabular records, examined its structure, ran basic exploratory analyses, and visualized data distributions.

You can extend this notebook by performing advanced analyses or applying machine learning techniques depending on your use case and research goals.